# Phase 10: Synapse Arbitrator Validation
Walk-Forward Testing (2020-2024) on Google Colab

**Objective:** Compare Synapse Arbitrator (Profit-Based) against baselines.

## 1. Setup

In [ ]:
# Upload colab_package.zip first, then run this cell
!unzip -o colab_package.zip -d /content/finrl_pro
%cd /content/finrl_pro
!pip install -q torch pandas numpy yfinance stockstats scipy optuna mlflow

In [ ]:
import sys
sys.path.insert(0, '/content/finrl_pro')

import numpy as np
import pandas as pd
import yfinance as yf
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("Setup complete.")

## 2. Configuration

In [ ]:
# Universe (20 Blue-Chip Stocks)
TICKERS = [
    "WMT", "KO", "PEP", "MCD", "NKE", "COST", "CL",
    "CSCO", "INTC", "ORCL", "IBM", "VZ", "T", "ADBE", "TXN",
    "BA", "CAT", "DIS", "PFE", "HON"
]

# Walk-Forward Windows
WINDOWS = [
    {"name": "W1", "train": ("2020-01-01", "2021-06-30"), "valid": ("2021-07-01", "2021-12-31"), "test": ("2022-01-01", "2022-06-30")},
    {"name": "W2", "train": ("2020-07-01", "2022-06-30"), "valid": ("2022-07-01", "2022-12-31"), "test": ("2023-01-01", "2023-06-30")},
    {"name": "W3", "train": ("2021-01-01", "2023-06-30"), "valid": ("2023-07-01", "2023-12-31"), "test": ("2024-01-01", "2024-06-30")},
]

# HPO Settings
HPO_TRIALS = 10  # Reduced for speed
TRAIN_TIMESTEPS = 30000  # Per specialist

print(f"Tickers: {len(TICKERS)}")
print(f"Windows: {len(WINDOWS)}")

## 3. Download Data

In [ ]:
print("Downloading data from Yahoo Finance...")
df_list = []
for tic in TICKERS:
    try:
        data = yf.download(tic, start="2020-01-01", end="2024-12-31", progress=False)
        data = data.reset_index()
        data['tic'] = tic
        data.columns = ['date', 'open', 'high', 'low', 'close', 'adj_close', 'volume', 'tic']
        df_list.append(data)
    except Exception as e:
        print(f"Error downloading {tic}: {e}")

df = pd.concat(df_list, ignore_index=True)
df['date'] = pd.to_datetime(df['date'])
print(f"Downloaded {len(df)} rows for {df['tic'].nunique()} tickers.")
df.head()

## 4. Feature Engineering

In [ ]:
from stockstats import StockDataFrame

def add_technical_indicators(df):
    """Add MACD, RSI, Bollinger Bands."""
    result = []
    for tic in df['tic'].unique():
        tic_df = df[df['tic'] == tic].copy()
        stock = StockDataFrame.retype(tic_df[['date', 'open', 'high', 'low', 'close', 'volume']].copy())
        tic_df['macd'] = stock['macd']
        tic_df['rsi_14'] = stock['rsi_14']
        tic_df['boll_ub'] = stock['boll_ub']
        tic_df['boll_lb'] = stock['boll_lb']
        result.append(tic_df)
    return pd.concat(result, ignore_index=True)

df = add_technical_indicators(df)
df = df.dropna()
print(f"After features: {len(df)} rows")
df.head()

## 5. Helper Functions

In [ ]:
from finrl_pro.agents.ppo import PPOAgent
from finrl_pro.envs.factory import make_pro_env
from finrl_pro.data.loader_pro import ProFeatureAssembler
from finrl_pro.execution.arbitrator import SynapseArbitrator

def slice_data(df, start, end):
    """Slice dataframe by date range."""
    mask = (df['date'] >= start) & (df['date'] <= end)
    return df[mask].copy()

def create_env(df_slice):
    """Create trading environment from data slice."""
    assembler = ProFeatureAssembler()
    features_cfg = {
        "stockstats_overrides": ["macd", "rsi_14", "boll_ub", "boll_lb"],
        "use_turbulence": False
    }
    asm = assembler.assemble_from_df(df=df_slice, features_cfg=features_cfg, dataset_hash="colab")
    env = make_pro_env(asm, initial_capital=1000000, reward_scaling=1e-4)
    return env, asm

def train_agent(env, timesteps=30000):
    """Train a PPO agent."""
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.shape[0]
    agent = PPOAgent(state_dim=state_dim, action_dim=action_dim, env=env, device="cuda")
    agent.train(total_timesteps=timesteps)
    return agent

def evaluate_agent(agent, env):
    """Evaluate agent and return metrics."""
    obs, _ = env.reset()
    done = False
    rewards = []
    while not done:
        action = agent.act(obs)
        obs, reward, done, truncated, info = env.step(action)
        rewards.append(reward)
        if truncated: done = True
    returns = np.array(rewards)
    sharpe = np.mean(returns) / (np.std(returns) + 1e-8) * np.sqrt(252)
    total_return = np.sum(returns)
    return {"sharpe": sharpe, "total_return": total_return, "steps": len(rewards)}

def evaluate_synapse(agents, env, profit_mode=True):
    """Evaluate Synapse Arbitrator."""
    arbitrator = SynapseArbitrator(agents, n_samples=100, window_size=20)
    obs, _ = env.reset()
    done = False
    rewards = []
    while not done:
        action, _ = arbitrator.predict(obs)
        obs, reward, done, truncated, info = env.step(action)
        rewards.append(reward)
        if profit_mode:
            arbitrator.update(reward)
        if truncated: done = True
    returns = np.array(rewards)
    sharpe = np.mean(returns) / (np.std(returns) + 1e-8) * np.sqrt(252)
    return {"sharpe": sharpe, "total_return": np.sum(returns)}

print("Helper functions loaded.")

## 6. Walk-Forward Loop

In [ ]:
results = []

for window in WINDOWS:
    print(f"\n{'='*50}")
    print(f"Processing {window['name']}")
    print(f"{'='*50}")
    
    # Slice Data
    train_df = slice_data(df, window['train'][0], window['train'][1])
    valid_df = slice_data(df, window['valid'][0], window['valid'][1])
    test_df = slice_data(df, window['test'][0], window['test'][1])
    
    print(f"Train: {len(train_df)} rows | Valid: {len(valid_df)} rows | Test: {len(test_df)} rows")
    
    # Create Environments
    train_env, _ = create_env(train_df)
    valid_env, _ = create_env(valid_df)
    test_env, _ = create_env(test_df)
    
    # Train Specialists (Simplified: Same data, different seeds)
    print("Training specialists...")
    specialists = []
    for i, name in enumerate(["Bull", "Bear", "Neutral"]):
        print(f"  Training {name} Specialist...")
        np.random.seed(42 + i)
        agent = train_agent(train_env, timesteps=TRAIN_TIMESTEPS)
        specialists.append(agent)
    
    # Evaluate on Test Set
    print("Evaluating on test set...")
    
    # Baseline: Single Best (First agent for simplicity)
    single_metrics = evaluate_agent(specialists[0], test_env)
    
    # Synapse: Profit Mode
    synapse_metrics = evaluate_synapse(specialists, test_env, profit_mode=True)
    
    # Record Results
    results.append({
        "window": window['name'],
        "test_period": f"{window['test'][0]} to {window['test'][1]}",
        "single_sharpe": single_metrics['sharpe'],
        "synapse_sharpe": synapse_metrics['sharpe'],
        "synapse_vs_single": synapse_metrics['sharpe'] - single_metrics['sharpe']
    })
    
    print(f"Single Best Sharpe: {single_metrics['sharpe']:.4f}")
    print(f"Synapse Sharpe: {synapse_metrics['sharpe']:.4f}")
    
print("\n" + "="*50)
print("WALK-FORWARD COMPLETE")
print("="*50)

## 7. Results Summary

In [ ]:
results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

# Global Metrics
print(f"\nGlobal Single Best Sharpe: {results_df['single_sharpe'].mean():.4f}")
print(f"Global Synapse Sharpe: {results_df['synapse_sharpe'].mean():.4f}")
print(f"Average Improvement: {results_df['synapse_vs_single'].mean():.4f}")

# Save Results
results_df.to_csv('phase10_synapse_results.csv', index=False)
print("\nResults saved to phase10_synapse_results.csv")

In [ ]:
# Plot Comparison
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(results_df))
width = 0.35

ax.bar([i - width/2 for i in x], results_df['single_sharpe'], width, label='Single Best', color='gray')
ax.bar([i + width/2 for i in x], results_df['synapse_sharpe'], width, label='Synapse (Profit)', color='green')

ax.set_xlabel('Window')
ax.set_ylabel('Sharpe Ratio')
ax.set_title('Phase 10: Synapse vs Single Best Agent')
ax.set_xticks(x)
ax.set_xticklabels(results_df['window'])
ax.legend()
ax.axhline(y=1.0, color='red', linestyle='--', label='Target (1.0)')

plt.tight_layout()
plt.savefig('phase10_synapse_comparison.png', dpi=150)
plt.show()
print("Chart saved to phase10_synapse_comparison.png")